In [151]:
from typing import Optional
from DATA.stock_invest_function import get_db_host
import pymysql
import pandas as pd

def fetch_fs_data_by_ticker(db_info: dict,
                            ticker: str,
                            table_name: str = "korea_fs_data_from_DART") -> pd.DataFrame:
    """
    특정 ticker의 재무제표 데이터를 DB에서 조회.
    ticker가 존재하지 않을 경우 메시지 출력 후 빈 DataFrame 반환.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        # 먼저 ticker 존재 여부 확인
        check_sql = f"SELECT COUNT(*) AS cnt FROM {table_name} WHERE ticker = %s"
        with conn.cursor() as cur:
            cur.execute(check_sql, (ticker,))
            result = cur.fetchone()
            cnt = result[0]

        if cnt == 0:
            print(f"[INFO] ticker '{ticker}' 는(은) 데이터베이스에 존재하지 않습니다.")
            return pd.DataFrame()   # 빈 DF 반환

        # ticker 존재 → 실제 데이터 조회
        query = f"""
            SELECT
                corp_code,
                bsns_year,
                reprt_code,
                quarter,
                account_id,
                sj_div,
                sj_nm,
                account_nm,
                thstrm_nm,
                thstrm_amount,
                report_date,
                ticker
            FROM {table_name}
            WHERE ticker = %s
            ORDER BY
                bsns_year,
                reprt_code,
                sj_div,
                account_nm
        """

        df = pd.read_sql(query, conn, params=[ticker])
        return df

    finally:
        conn.close()

def adjust_quarterly_from_index(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    """
    단일 종목 분기 데이터에서,
    12월 값(FY 누적)을 4분기 순액으로 전환:
        4Q = FY - (Q1 + Q2 + Q3)

    전제:
      - df의 index = 날짜 (DatetimeIndex)
      - value_cols = 분기 조정할 숫자 컬럼 리스트
    """

    df = df.copy()

    # 1) 인덱스를 날짜로 강제
    df.index = pd.to_datetime(df.index)
    original_index_name = df.index.name  # 나중에 복원용

    # 2) 연도 / 분기 정보 생성 (index만 사용, 'report_date' 컬럼 안 만듦)
    df["year"] = df.index.year
    df["quarter"] = df.index.month.map({3: "Q1", 6: "Q2", 9: "Q3", 12: "Q4"})

    adjusted_chunks = []

    # 3) 연도별로 그룹 나눠서 4Q 조정
    for year, grp in df.groupby("year"):
        grp = grp.sort_index()

        q1 = grp[grp["quarter"] == "Q1"]
        q2 = grp[grp["quarter"] == "Q2"]
        q3 = grp[grp["quarter"] == "Q3"]
        q4 = grp[grp["quarter"] == "Q4"]

        if len(q4) > 0:
            q4 = q4.copy()

            for col in value_cols:
                if col not in grp.columns:
                    continue  # 없는 컬럼은 무시

                # FY 값 (12월 한 줄이라고 가정)
                fy = q4[col].iloc[0]
                if pd.isna(fy):
                    continue  # FY 자체가 NaN이면 건드리지 않음

                prev_sum = (
                    q1[col].fillna(0).sum()
                    + q2[col].fillna(0).sum()
                    + q3[col].fillna(0).sum()
                )

                q4[col] = fy - prev_sum

            adjusted_chunks.extend([q1, q2, q3, q4])
        else:
            # 4Q가 없는 연도는 그대로
            adjusted_chunks.append(grp)

    # 4) 다시 합치고 정리
    result = pd.concat(adjusted_chunks).sort_index()
    result = result.drop(columns=["year", "quarter"])

    # 인덱스 이름 복원
    result.index.name = original_index_name

    return result


In [152]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}

df = fetch_fs_data_by_ticker(db_info, "000660")

if df.empty:
    print("데이터 없음")
else:
    display(df.head())


,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker
0,00164779,2015,11011,FY,ifrs_InvestmentAccountedForUsingEquityMethod,BS,재무상태표,관계기업 및 공동기업투자,제 68 기,1.226090e+11,2015-12-31,000660
1,00164779,2015,11011,FY,ifrs_OtherCurrentFinancialLiabilities,BS,재무상태표,기타금융부채,제 68 기,NaN,2015-12-31,000660
2,00164779,2015,11011,FY,ifrs_OtherNoncurrentFinancialLiabilities,BS,재무상태표,기타금융부채,제 68 기,6.830000e+08,2015-12-31,000660
3,00164779,2015,11011,FY,ifrs_OtherCurrentFinancialAssets,BS,재무상태표,기타금융자산,제 68 기,NaN,2015-12-31,000660
4,00164779,2015,11011,FY,ifrs_OtherNoncurrentFinancialAssets,BS,재무상태표,기타금융자산,제 68 기,4.300000e+08,2015-12-31,000660


In [153]:
IS = df[df['sj_div'] == 'IS']
BS = df[df['sj_div'] == 'BS']


In [155]:
is_table['매출액_수정'] = is_table['매출액'].fillna(is_table["수익(매출액)"])
is_table['영업이익_수정'] = is_table["영업이익"].fillna(is_table["영업이익(손실)"])
is_table['기타이익_수정'] = is_table["기타이익"].fillna(is_table["기타수익"])
is_table['기타손실_수정'] = is_table["기타손실"].fillna(is_table["기타비용"])
is_table['법인세차감전순이익_수정'] = is_table["법인세비용차감전순이익"].fillna(is_table["법인세비용차감전순이익(손실)"])
is_table['법인세비용_수정'] = is_table["법인세비용"].fillna(is_table["법인세비용(수익)"])
is_table['당기순이익_비지배주주_수정'] = is_table["비지배지분"].fillna(is_table["비지배지분에 귀속되는 당기순이익(손실)"])
# 매출액 최종 수정
is_table["매출액_수정"] = is_table["매출액_수정"].fillna(is_table["매출원가"] + is_table["매출총이익"])

# 결합 대상 컬럼 목록
cols = [
    '분기순이익',
    '지배기업 소유주지분',
    '지배기업 소유지분',
    '지배기업의 소유주에게 귀속되는 당기순이익(손실)'
]

# 존재하는 컬럼만 필터링
available_cols = [c for c in cols if c in is_table.columns]

if not available_cols:
    print("[WARN] 결합할 칼럼이 없습니다.")
else:
    # row-wise 로 결합 → 첫 번째로 NaN이 아닌 값을 선택
    is_table['당기순이익_지배주주'] = is_table[available_cols].bfill(axis=1).iloc[:, 0]

is_table['당기순이익'] = is_table['당기순이익_지배주주'] + is_table['당기순이익_비지배주주_수정']

value_cols = [
    "매출액_수정",
    "매출원가",
    "매출총이익",
    "판매비와관리비",
    "영업이익_수정",
    "금융비용",
    "금융수익",
    "기타손익_수정",
    "법인세비용차감전순이익_수정",
    "법인세비용_수정",
    "당기순이익_지배주주",
    "당기순이익_비지배주주_수정",
    "당기순이익"
]

is_table_adj = adjust_quarterly_from_index(is_table, value_cols)


In [156]:
is_table_adj

,계속영업이익(손실),금융비용,금융수익,기본주당이익,기본주당이익(손실),기본주당이익(손실) (단위:원),기타비용,기타손실,기타수익,기타이익,...,ticker,매출액_수정,영업이익_수정,기타이익_수정,기타손실_수정,법인세차감전순이익_수정,법인세비용_수정,당기순이익_지배주주,당기순이익_비지배주주_수정,당기순이익
report_date,,,,,,,,,,,,,,,,,,,,,
2015-12-31,1.906014e+13,1.003177e+13,1.051488e+13,NaN,NaN,126305.0,3.723434e+12,NaN,1.685947e+12,NaN,...,005930,2.006535e+14,2.641344e+13,1.685947e+12,3.723434e+12,2.596100e+13,6.900851e+12,1.869463e+13,3.655160e+11,1.906014e+13
2016-03-31,5.252803e+12,3.006360e+12,3.079929e+12,NaN,NaN,36356.0,3.386360e+11,NaN,7.025930e+11,NaN,...,005930,4.978225e+13,6.675812e+12,7.025930e+11,3.386360e+11,7.122485e+12,1.869682e+12,5.263506e+12,-1.070300e+10,5.252803e+12
2016-06-30,5.847393e+12,1.822601e+12,1.986063e+12,NaN,NaN,40904.0,5.154530e+11,NaN,2.757240e+11,NaN,...,005930,5.093712e+13,8.143950e+12,2.757240e+11,5.154530e+11,8.071959e+12,2.224566e+12,5.826178e+12,2.121500e+10,5.847393e+12
2016-09-30,4.537855e+12,2.075981e+12,2.303768e+12,NaN,NaN,31303.0,5.457210e+11,NaN,1.083572e+12,NaN,...,005930,4.781563e+13,5.200089e+12,1.083572e+12,5.457210e+11,5.970702e+12,1.432847e+12,4.408790e+12,1.290650e+11,4.537855e+12
2016-12-31,2.272609e+13,3.801671e+12,4.015885e+12,NaN,NaN,157967.0,2.463814e+12,NaN,3.238261e+12,NaN,...,005930,5.333174e+13,9.220821e+12,3.238261e+12,2.463814e+12,3.071365e+13,2.460465e+12,6.917181e+12,1.708600e+11,7.088041e+12
2017-03-31,7.684354e+12,1.889464e+12,2.097152e+12,NaN,NaN,53656.0,2.772340e+11,NaN,3.164230e+11,NaN,...,005930,5.054753e+13,9.898361e+12,3.164230e+11,2.772340e+11,1.016458e+13,2.480222e+12,7.488532e+12,1.958220e+11,7.684354e+12
2017-06-30,1.105385e+13,1.524619e+12,1.727174e+12,NaN,NaN,78018.0,2.234050e+11,NaN,3.442150e+11,NaN,...,005930,6.100054e+13,1.406655e+13,3.442150e+11,2.234050e+11,1.441226e+13,3.358414e+12,1.079994e+13,2.539090e+11,1.105385e+13
2017-09-30,1.119341e+13,NaN,2.610849e+12,NaN,NaN,80349.0,2.781220e+11,NaN,3.757740e+11,NaN,...,005930,6.204890e+13,1.453316e+13,3.757740e+11,2.781220e+11,1.491474e+13,3.721329e+12,1.103977e+13,1.536400e+11,1.119341e+13
2017-12-31,4.218675e+13,NaN,3.302216e+12,NaN,NaN,299868.0,1.419648e+12,NaN,3.010657e+12,NaN,...,005930,6.597841e+13,1.514697e+13,3.010657e+12,1.419648e+12,5.619597e+13,4.449255e+12,1.201632e+13,2.388070e+11,1.225513e+13


In [126]:
is_table[['매출액_수정', '매출원가', '매출총이익', '판매비와관리비', '영업이익_수정', '금융비용', '금융수익',
          '기타이익_수정', '기타손실_수정', '법인세차감전순이익_수정', '법인세비용_수정']]

,매출액_수정,매출원가,매출총이익,판매비와관리비,영업이익_수정,금융비용,금융수익,기타이익_수정,기타손실_수정,법인세차감전순이익_수정,법인세비용_수정
report_date,,,,,,,,,,,
2015-12-31,2.006535e+14,1.234821e+14,7.717136e+13,5.075792e+13,2.641344e+13,1.003177e+13,1.051488e+13,1.685947e+12,3.723434e+12,2.596100e+13,6.900851e+12
2016-03-31,4.978225e+13,3.037386e+13,1.940839e+13,1.273258e+13,6.675812e+12,3.006360e+12,3.079929e+12,7.025930e+11,3.386360e+11,7.122485e+12,1.869682e+12
2016-06-30,5.093712e+13,2.960912e+13,2.132800e+13,1.318405e+13,8.143950e+12,1.822601e+12,1.986063e+12,2.757240e+11,5.154530e+11,8.071959e+12,2.224566e+12
2016-09-30,4.781563e+13,2.941126e+13,1.840438e+13,1.320429e+13,5.200089e+12,2.075981e+12,2.303768e+12,1.083572e+12,5.457210e+11,5.970702e+12,1.432847e+12
2016-12-31,2.018667e+14,1.202777e+14,8.158903e+13,5.234836e+13,2.924067e+13,1.070661e+13,1.138564e+13,3.238261e+12,2.463814e+12,3.071365e+13,7.987560e+12
2017-03-31,5.054753e+13,2.815560e+13,2.239193e+13,1.249357e+13,9.898361e+12,1.889464e+12,2.097152e+12,3.164230e+11,2.772340e+11,1.016458e+13,2.480222e+12
2017-06-30,6.100054e+13,3.239983e+13,2.860071e+13,1.453416e+13,1.406655e+13,1.524619e+12,1.727174e+12,3.442150e+11,2.234050e+11,1.441226e+13,3.358414e+12
2017-09-30,6.204890e+13,3.300412e+13,2.904478e+13,1.451162e+13,1.453316e+13,NaN,2.610849e+12,3.757740e+11,2.781220e+11,1.491474e+13,3.721329e+12
2017-12-31,2.395754e+14,1.292907e+14,1.102847e+14,5.663968e+13,5.364504e+13,NaN,9.737391e+12,3.010657e+12,1.419648e+12,5.619597e+13,1.400922e+13


In [67]:
wanted_cols = [
    '매출액',
    '당기순이익(손실)',
    '분기순이익',
    '반기순이익',
    '반기총포괄이익',
    '영업이익',
    '분기총포괄손익',
    '분기총포괄이익',
    '분기총포괄이익(손실)',
    '총포괄손익'
]

# 실제 pivot_table 에 존재하는 컬럼만 필터링
existing_cols = [c for c in wanted_cols if c in pivot_table.columns]

missing_cols = [c for c in wanted_cols if c not in pivot_table.columns]
if missing_cols:
    print("[INFO] 다음 컬럼은 pivot_table에 없습니다:", missing_cols)

pivot_table[existing_cols].tail(12)

[INFO] 다음 컬럼은 pivot_table에 없습니다: ['반기총포괄이익', '분기총포괄이익', '분기총포괄이익(손실)']


,매출액,당기순이익(손실),분기순이익,반기순이익,영업이익,분기총포괄손익,총포괄손익
report_date,,,,,,,
2022-12-31,NaN,9.240590e+11,NaN,NaN,4.337663e+13,NaN,5.965974e+13
2023-03-31,NaN,1.733480e+11,NaN,NaN,6.401780e+11,NaN,7.554100e+12
2023-06-30,NaN,3.499010e+11,NaN,NaN,6.685470e+11,NaN,1.639827e+12
2023-09-30,NaN,8.449574e+12,NaN,NaN,2.433534e+12,NaN,7.417032e+12
2023-12-31,NaN,1.447340e+13,NaN,NaN,6.566976e+12,NaN,NaN
2024-03-31,7.191560e+13,NaN,NaN,NaN,6.606009e+12,1.174966e+13,NaN
2024-06-30,7.406830e+13,NaN,NaN,0.000000e+00,1.044388e+13,NaN,NaN
2024-09-30,7.909873e+13,NaN,0.000000e+00,NaN,9.183371e+12,5.255744e+12,NaN
2024-12-31,3.008709e+14,NaN,NaN,NaN,3.272596e+13,NaN,5.129634e+13


In [63]:
pivot_table.columns.tolist()

['계속영업이익(손실)',
 '관계기업 및 공동기업 투자',
 '관계기업 및 공동기업 투자의 처분',
 '관계기업 및 공동기업 투자의 취득',
 '관계기업 및 공동기업의 기타포괄손익에 대한 지분',
 '관계종속기업투자자산-지분법',
 '금융비용',
 '금융수익',
 '기말 현금 및 현금성자산',
 '기말 현금및현금성자산',
 '기말의 현금및현금성자산',
 '기말현금및현금성자산',
 '기본주당이익',
 '기본주당이익(손실)',
 '기본주당이익(손실) (단위:원)',
 '기초 현금 및 현금성자산',
 '기초 현금및현금성자산',
 '기초의 현금및현금성자산',
 '기초자본',
 '기초현금및현금성자산',
 '기타',
 '기타 비유동 부채',
 '기타 유동부채',
 '기타거래',
 '기타비용',
 '기타비유동부채',
 '기타비유동자산',
 '기타손실',
 '기타수익',
 '기타유동부채',
 '기타유동자산',
 '기타이익',
 '기타자본항목',
 '기타투자활동으로 인한 현금유출입액',
 '기타포괄손익',
 '기타포괄손익(*4)',
 '기타포괄손익-공정가치 측정 금융자산 평가손익',
 '기타포괄손익-공정가치 측정 비유동금융자산',
 '기타포괄손익-공정가치금융자산',
 '기타포괄손익-공정가치금융자산의 처분',
 '기타포괄손익-공정가치금융자산의 취득',
 '기타포괄손익-공정가치금융자산평가손익',
 '기타포괄손익-공정가치측정금융자산의처분',
 '기타포괄손익-공정가치측정금융자산의취득',
 '단기금융상품',
 '단기당기손익-공정가치금융자산',
 '단기매도가능금융자산의 처분',
 '단기매도가능금융자산의 취득',
 '단기상각후원가금융자산',
 '단기차입금',
 '단기차입금의 순증가(감소)',
 '단기차입금의 순증가(감소) (주27)',
 '당기법인세부채',
 '당기손익-공정가치금융자산',
 '당기손익-공정가치금융자산의 처분',
 '당기손익-공정가치금융자산의 취득',
 '당기손익으로 재분류되는 세후기타포괄손익',
 '당기순이익',
 '당기순이익(손실)',
 '만기보유금융자산의 취득',
 

In [117]:
is_table[['비지배지분', '비지배지분에 귀속되는 당기순이익(손실)',]]

,비지배지분,비지배지분에 귀속되는 당기순이익(손실)
report_date,,
2015-12-31,NaN,3.655160e+11
2016-03-31,NaN,-1.070300e+10
2016-06-30,NaN,2.121500e+10
2016-09-30,NaN,1.290650e+11
2016-12-31,NaN,3.104370e+11
2017-03-31,NaN,1.958220e+11
2017-06-30,NaN,2.539090e+11
2017-09-30,NaN,1.536400e+11
2017-12-31,NaN,8.421780e+11


In [43]:
pivot_table.columns.tolist()

['관계기업 기타포괄손익지분',
 '관계기업 및 공동기업투자',
 '관계기업에 대한 투자자산의 취득',
 '관계기업의 기타포괄손익에 대한 지분',
 '관계기업투자의 처분',
 '관계기업투자의 취득',
 '관계기업투자주식의 처분',
 '관계기업투자주식의 취득',
 '금융비용',
 '금융수익',
 '기말 현금및현금성자산',
 '기말현금및현금성자산',
 '기본주당반기순이익',
 '기본주당반기순이익(손실)',
 '기본주당분기순이익',
 '기본주당분기순이익(손실)',
 '기본주당순이익',
 '기본주당이익(손실)',
 '기본주당이익(원)',
 '기초 현금및현금성자산',
 '기초자본',
 '기초현금및현금성자산',
 '기타',
 '기타 비유동 부채',
 '기타 유동부채',
 '기타 투자활동으로 인한 현금유출입',
 '기타금융부채',
 '기타금융자산',
 '기타금융자산의 감소',
 '기타금융자산의 증가',
 '기타금융자산의 처분',
 '기타금융자산의 처분(취득)',
 '기타금융자산의 취득',
 '기타비유동부채',
 '기타비유동자산',
 '기타수취채권',
 '기타수취채권의 감소',
 '기타수취채권의 증가',
 '기타영업외비용',
 '기타영업외수익',
 '기타유동부채',
 '기타유동자산',
 '기타자본',
 '기타지급채무',
 '기타포괄손익',
 '기타포괄손익누계액',
 '단기금융상품',
 '단기금융상품의 감소',
 '단기금융상품의 순증감',
 '단기금융상품의 증가',
 '단기미지급금',
 '단기차입금 및 유동성 장기차입금',
 '단기투자자산',
 '단기투자자산의 순증감',
 '당기법인세부채',
 '당기법인세자산',
 '당기손익으로 재분류되는 세후기타포괄손익',
 '당기손익으로 재분류되지 않는 세후기타포괄손익',
 '당기순이익',
 '당기순이익(손실)',
 '리스부채',
 '리스부채의 상환',
 '매각예정부채',
 '매각예정자산',
 '매각예정자산의 처분',
 '매도가능금융자산',
 '매도가능금융자산의 처분',
 '매도가능금융자산의 취득',
 '매도가능금융자산평가손익',
 '매입

In [82]:
temp = df[(df['account_id'] == 'ifrs-full_ProfitLoss') | (df['account_id'] == 'ifrs_ProfitLoss') | (df['account_id'] == 'ifrs-full_ComprehensiveIncome')]

temp_profit = temp[temp['sj_div'] == 'IS']

In [83]:
temp_profit

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker


In [84]:
df[df['sj_div'] == 'IS']

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker
64,00126380,2015,11011,FY,ifrs_ProfitLossFromContinuingOperations,IS,손익계산서,계속영업이익(손실),제 47 기,1.906014e+13,2015-12-31,005930
65,00126380,2015,11011,FY,ifrs_FinanceCosts,IS,손익계산서,금융비용,제 47 기,1.003177e+13,2015-12-31,005930
66,00126380,2015,11011,FY,ifrs_FinanceIncome,IS,손익계산서,금융수익,제 47 기,1.051488e+13,2015-12-31,005930
67,00126380,2015,11011,FY,ifrs_BasicEarningsLossPerShare,IS,손익계산서,기본주당이익(손실) (단위:원),제 47 기,1.263050e+05,2015-12-31,005930
68,00126380,2015,11011,FY,dart_OtherLosses,IS,손익계산서,기타비용,제 47 기,3.723434e+12,2015-12-31,005930
...,...,...,...,...,...,...,...,...,...,...,...,...
3916,00126380,2025,11014,Q3,dart_OperatingIncomeLoss,IS,손익계산서,영업이익,제 57 기 3분기,1.216606e+13,2025-09-30,005930
3917,00126380,2025,11014,Q3,ifrs-full_ProfitLossAttributableToOwnersOfParent,IS,손익계산서,지배기업 소유주지분,제 57 기 3분기,1.200646e+13,2025-09-30,005930
3918,00126380,2025,11014,Q3,ifrs-full_ShareOfProfitLossOfAssociatesAndJoin...,IS,손익계산서,지분법이익,제 57 기 3분기,2.804880e+11,2025-09-30,005930
3919,00126380,2025,11014,Q3,dart_TotalSellingGeneralAdministrativeExpenses,IS,손익계산서,판매비와관리비,제 57 기 3분기,2.132606e+13,2025-09-30,005930


In [72]:
df['account_id'].unique().tolist()

['ifrs_InvestmentAccountedForUsingEquityMethod',
 'dart_ElementsOfOtherStockholdersEquity',
 'dart_ShortTermBorrowings',
 'ifrs_LiabilitiesIncludedInDisposalGroupsClassifiedAsHeldForSale',
 'ifrs_NoncurrentAssetsOrDisposalGroupsClassifiedAsHeldForSaleOrAsHeldForDistributionToOwners',
 'dart_PaymentsOfIncomeTaxesPayable',
 'ifrs_Liabilities',
 'ifrs_NoncurrentLiabilities',
 'ifrs_NoncurrentAssets',
 'ifrs_NoncontrollingInterests',
 'dart_PostemploymentBenefitObligations',
 'ifrs_CurrentLiabilities',
 'ifrs_CurrentPortionOfLongtermBorrowings',
 'ifrs_CurrentAssets',
 'ifrs_PropertyPlantAndEquipment',
 'ifrs_DeferredTaxLiabilities',
 'ifrs_DeferredTaxAssets',
 'ifrs_RetainedEarnings',
 'ifrs_EquityAndLiabilities',
 'ifrs_Equity',
 'ifrs_Assets',
 'dart_LongTermBorrowingsGross',
 'ifrs_NoncurrentProvisions',
 'ifrs_Inventories',
 'ifrs_EquityAttributableToOwnersOfParent',
 'ifrs_CurrentProvisions',
 'ifrs_CashAndCashEquivalents',
 'dart_CashAndCashEquivalentsAtEndOfPeriodCf',
 'dart_CashAn

In [78]:
df[df['account_nm'].str.contains('분기')]

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker
3420,00126380,2024,11013,Q1,ifrs-full_ComprehensiveIncome,CIS,포괄손익계산서,분기총포괄손익,제 56 기 1분기,1.174966e+13,2024-03-31,005930
3446,00126380,2024,11013,Q1,ifrs-full_ProfitLoss,SCE,자본변동표,분기순이익(손실),제 56 기 1분기,0.000000e+00,2024-03-31,005930
3539,00126380,2024,11014,Q3,ifrs-full_ComprehensiveIncome,CIS,포괄손익계산서,분기총포괄손익,제 56 기 3분기,5.255744e+12,2024-09-30,005930
3565,00126380,2024,11014,Q3,ifrs-full_ProfitLoss,SCE,자본변동표,분기순이익,제 56 기 3분기,0.000000e+00,2024-09-30,005930
3755,00126380,2025,11013,Q1,dart_CashAndCashEquivalentsAtEndOfPeriodCf,CF,현금흐름표,분기말의 현금및현금성자산,제 57 기 1분기,5.316100e+13,2025-03-31,005930
3792,00126380,2025,11013,Q1,ifrs-full_ProfitLossAttributableToOwnersOfParent,IS,손익계산서,분기순이익,제 57 기 1분기,8.028407e+12,2025-03-31,005930
3803,00126380,2025,11013,Q1,ifrs-full_ProfitLoss,SCE,자본변동표,분기순이익,제 57 기 1분기,8.028407e+12,2025-03-31,005930
3877,00126380,2025,11014,Q3,dart_CashAndCashEquivalentsAtEndOfPeriodCf,CF,현금흐름표,분기말의 현금및현금성자산,제 57 기 3분기,5.339948e+13,2025-09-30,005930
3900,00126380,2025,11014,Q3,ifrs-full_ComprehensiveIncome,CIS,포괄손익계산서,분기총포괄손익,제 57 기 3분기,2.017141e+13,2025-09-30,005930
3926,00126380,2025,11014,Q3,ifrs-full_ProfitLoss,SCE,자본변동표,분기순이익,제 57 기 3분기,2.496890e+13,2025-09-30,005930


In [9]:
def sample_trade_payables(db_info, table_name="korea_fs_data_from_DART"):
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        sql = f"""
            SELECT date, company_name, ticker, indicator, value
            FROM {table_name}
            WHERE indicator = 'Trade_Payables'
            ORDER BY ticker, date
            LIMIT 50
        """
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    return df

df_tp = sample_trade_payables(db_info)
df_tp.head()

,date,company_name,ticker,indicator,value
0,date,company_name,ticker,indicator,value
1,date,company_name,ticker,indicator,value
2,date,company_name,ticker,indicator,value
3,date,company_name,ticker,indicator,value
4,date,company_name,ticker,indicator,value


In [12]:
def list_all_indicators(db_info,
                        table_name="korea_fs_data_from_DART"):
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        sql = f"""
            SELECT DISTINCT indicator
            FROM {table_name}
            WHERE indicator IS NOT NULL
              AND indicator <> ''
              AND indicator <> 'indicator'
        """
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    indicators = sorted(df["indicator"].dropna().unique().tolist())
    print(f"📊 총 indicator 개수: {len(indicators)}")
    print(f"✅ ROA 존재 여부: {'ROA' in indicators}")
    print(f"✅ ROE 존재 여부: {'ROE' in indicators}")
    return indicators

all_indicators = list_all_indicators(db_info)

📊 총 indicator 개수: 1
✅ ROA 존재 여부: False
✅ ROE 존재 여부: False


In [13]:
all_indicators = list_all_indicators(db_info)
"ROA" in all_indicators, "ROE" in all_indicators

📊 총 indicator 개수: 1
✅ ROA 존재 여부: False
✅ ROE 존재 여부: False


(False, False)